In [ ]:
# default_exp paper.cnn.learning_curve

# 4.2. Learning curve (CNN)

> Computing the learning curve of the CNN for the prediction of exchangeable potassium (K ex.), with increasing number of training examples and using all Soil Taxonomy Orders. 

In [ ]:
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive',  force_remount=False)
    !pip install mirzai
else:
    %load_ext autoreload
    %autoreload 2

In [ ]:
# Python utils
import math
from collections import OrderedDict
from tqdm.auto import tqdm
from pathlib import Path

# mirzai utilities
from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import log_transform_y
from mirzai.training.cnn import (Model, weights_init)
from mirzai.data.torch import DataLoaders, SNV_transform
from mirzai.training.cnn import Learner

from fastcore.transform import compose

# Data science stack
import numpy as np
from sklearn.model_selection import train_test_split

# Deep Learning stack
import torch
from torch.nn import MSELoss
from torch.optim import Adam
from torch.optim.lr_scheduler import CyclicLR

import warnings
warnings.filterwarnings('ignore')

## Load and transform

In [ ]:
src_dir = '../_data'
fnames = ['spectra-features.npy', 'spectra-wavenumbers.npy', 
          'depth-order.npy', 'target.npy', 
          'tax-order-lu.pkl', 'spectra-id.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)

data = X, y, X_id, depth_order

transforms = [select_y, select_tax_order, select_X, log_transform_y]
X, y, X_id, depth_order = compose(*transforms)(data)

In [ ]:
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Wavenumbers:\n {X_names}')
print(f'depth_order (first 3 rows):\n {depth_order[:3, :]}')
print(f'Taxonomic order lookup:\n {tax_lookup}')

X shape: (40132, 1764)
y shape: (40132,)
Wavenumbers:
 [3999 3997 3995 ...  603  601  599]
depth_order (first 3 rows):
 [[43.  2.]
 [ 0.  0.]
 [ 0.  1.]]
Taxonomic order lookup:
 {'alfisols': 0, 'mollisols': 1, 'inceptisols': 2, 'entisols': 3, 'spodosols': 4, 'undefined': 5, 'ultisols': 6, 'andisols': 7, 'histosols': 8, 'oxisols': 9, 'vertisols': 10, 'aridisols': 11, 'gelisols': 12}


## Experiment

### Setup

In [ ]:
# Is a GPU available?
use_cuda = torch.cuda.is_available()
device = torch.device('cuda:0' if use_cuda else 'cpu')
print(f'Runtime is: {device}')

Runtime is: cpu


In [ ]:
history = OrderedDict({'nb_samples': [], 'train_score': [], 'valid_score': [], 'nb_epochs': []})
training_size = [100, 500, 1000, 2500, 5000, 10000, 20000, 30000, X.shape[0]]    
criterion = MSELoss() # Mean Squared Error loss
target_mse = 0.01
n_epochs_max = 5000
step_size_up = 5
base_lr, max_lr = 3e-5, 1e-3 # Based on Learning Rate

# If no GPU then just for test
if device.type == 'cpu':
    training_size = [100, 500]
    n_epochs_max = 2

### Run

In [ ]:
for size in tqdm(training_size):
    print(100*'-')
    print('# of samples: {}'.format(size))

    idx = np.random.choice(len(X), size, replace=False)
    
    data = train_test_split(X[idx, :], y[idx], test_size=0.2, random_state=42)
    X_train, X_test, y_train, y_test = data
    dls = DataLoaders(((X_train, y_train), (X_test, y_test)), transform=SNV_transform())
    training_generator, validation_generator = dls.loaders()
    
    model = Model(X_train.shape[1], out_channel=16).to(device)

    opt = Adam(model.parameters(), lr=1e-4)
    model = model.apply(weights_init)

    scheduler = CyclicLR(opt, base_lr=base_lr, max_lr=max_lr,
                         step_size_up=step_size_up, mode='triangular',
                         cycle_momentum=False)
        
    learner = Learner(model, criterion, opt, n_epochs=n_epochs_max, 
                      scheduler=scheduler, verbose=True)
    model, losses = learner.fit(training_generator, validation_generator)
    
    train_score = np.mean(losses['train'][-1])
    valid_score = np.mean(losses['valid'][-1])

    print(100*'-')
    print("# of samples: ", len(X_train))
    print("Train score: ", train_score)
    print("Valid score: ", valid_score)
    print("# of epochs", len(losses['train']))
    print(100*'-')

    history['nb_samples'].append(len(X_train))
    history['train_score'].append(train_score)
    history['valid_score'].append(valid_score)
    history['nb_epochs'].append(len(losses['train']))

  0%|          | 0/2 [00:00<?, ?it/s]

----------------------------------------------------------------------------------------------------
# of samples: 100
Runtime is: cpu


  0%|          | 0/2 [00:00<?, ?it/s]

End of Epoch 1
 Training loss: 0.18517172833283743
 Validation loss: 0.14987590909004211
End of Epoch 2
 Training loss: 0.1804222365220388
 Validation loss: 0.14882060885429382
----------------------------------------------------------------------------------------------------
# of samples:  80
Train score:  0.1804222365220388
Valid score:  0.14882060885429382
# of epochs 2
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
# of samples: 500
Runtime is: cpu


  0%|          | 0/2 [00:00<?, ?it/s]

End of Epoch 1
 Training loss: 0.18098111966481575
 Validation loss: 0.1472200332209468
End of Epoch 2
 Training loss: 0.17235723596352798
 Validation loss: 0.1438760426826775
----------------------------------------------------------------------------------------------------
# of samples:  400
Train score:  0.17235723596352798
Valid score:  0.1438760426826775
# of epochs 2
----------------------------------------------------------------------------------------------------


In [ ]:
history

OrderedDict([('nb_samples', [80, 400]),
             ('train_score', [0.1804222365220388, 0.17235723596352798]),
             ('valid_score', [0.14882060885429382, 0.1438760426826775]),
             ('nb_epochs', [2, 2])])

## Save

In [ ]:
MONITORING_PATH = Path('nameofyourfolder')
with open(MONITORING_PATH/'history_cnn.pickle'), 'wb') as f: 
    pickle.dump(history, f)